# Imbalanced Data

In [2]:
import gzip
import json
import pickle  ## to save our model as a file

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

## Prepare Data

### Import

Complete the wrangle function below using the code you developed in the last lesson. Then use it to import poland-bankruptcy-data-2009.json.gz 
into the DataFrame df.

In [ ]:
def wrangle(filename):
    # Open compressed file, load into dictionary
    with gzip.open(filename, "r") as f:
        data = json.load(f)

    # Load dictionary into Dataframe and set index
    df = pd.DataFrame().from_dict(data["data"]).set_index("company_id")

    return df

In [ ]:
df = wrangle("data/poland-bankruptcy-data-2009.json.gz")
print(df.shape)
df.head()

## Explore

In [ ]:
# Plot class balance
df["bankrupt"].value_counts(normalize=True).plot(
    kind="bar",
    xlabel = "Bankrupt",
    ylabel = "Frequency",
    title = "Class Balance"
)

In [ ]:
### Use seaborn to create a boxplot that shows the distributions of the "feat_27" column for both groups in the "bankrupt" column. 
## Remember to label your axes.

# Create boxplot
sns.boxplot(x="bankrupt", y="feat_27", data=df)



plt.xlabel("Bankrupt")
plt.ylabel("POA / financial expenses")
plt.title("Distribution of Profit/Expenses Ratio, by Class");

In [ ]:
## Use the describe method on the column for "feat_27". What can you tell about the distribution of the data based on the mean and median?

# Summary statistics for `feat_27`
df["feat_27"].describe().apply("{0:,.0f}".format)


# Note that the median is around 1, but the mean is over 1000. That suggests that this feature is skewed to the right. 
## Let's make a histogram to see what the distribution actually looks like.

In [ ]:
## Create a histogram of "feat_27". Make sure to label x-axis "POA / financial expenses", 
## the y-axis "Count", and use the title "Distribution of Profit/Expenses Ratio".

# Plot histogram of `feat_27`
df["feat_27"].hist()

plt.xlabel("POA / financial expenses")
plt.ylabel("Count"),
plt.title("Distribution of Profit/Expenses Ratio");

## We saw it in the numbers and now we see it in the histogram. The data is very skewed. So, in order to create a helpful boxplot, 
## we need to trim the data.

In [ ]:
## Recreate the boxplot that you made above, this time only using the values for "feat_27" that fall between the 0.1 and 0.9 quantiles for the column.

# Create clipped boxplot
q1, q9 = df["feat_27"].quantile([0.1, 0.9])
mask = df["feat_27"].between(q1, q9)
sns.boxplot(x= "bankrupt", y="feat_27", data=df[mask])
plt.xlabel("Bankrupt")
plt.ylabel("POA / financial expenses")
plt.title("Distribution of Profit/Expenses Ratio, by Bankruptcy Status");

In [ ]:
## Plot a correlation heatmap of features in df. Since "bankrupt" will be your target, you don't need to include it in your heatmap.

corr = df.drop(columns="bankrupt").corr()
sns.heatmap(corr)


## Split

In [ ]:
## Create your feature matrix X and target vector y. Your target is "bankrupt".

target = "bankrupt"
X = df.drop(columns=target)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
## Divide your data (X and y) into training and test sets using a randomized train-test split. Your validation set should be 20% of your total data. 
## And don't forget to set a random_state for reproducibility.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## Resample

In [ ]:
## Create a new feature matrix X_train_under and target vector y_train_under by performing random under-sampling on your training data.

under_sampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under_sampler.fit_resample(X_train, y_train)
print(X_train_under.shape)
X_train_under.head()

In [ ]:
## Create a new feature matrix X_train_over and target vector y_train_over by performing random over-sampling on your training data.

over_sampler = RandomOverSampler(random_state=42)
X_train_over, y_train_over = over_sampler.fit_resample(X_train, y_train)
print(X_train_over.shape)
X_train_over.head()

## Build Model

### Baseline

As always, we need to establish the baseline for our model. Since this is a classification problem, we'll use accuracy score.

In [ ]:
## Calculate the baseline accuracy score for your model

acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 4))

Baseline Accuracy: 0.9519
Note here that, because our classes are imbalanced, the baseline accuracy is very high. We should keep this in mind because, 
even if our trained model gets a high validation accuracy score, that doesn't mean it's actually good.

## Iterate
Create three identical models: model_reg, model_under and model_over. All of them should use a SimpleImputer followed by a DecisionTreeClassifier. 
Train model_reg using the unaltered training data. For model_under, use the undersampled data. For model_over, use the oversampled data.

In [ ]:
# Fit on `X_train`, `y_train`

model_reg = make_pipeline(
    SimpleImputer(strategy="median"), DecisionTreeClassifier(random_state=42)
)
model_reg.fit(X_train, y_train)  

# Fit on `X_train_under`, `y_train_under`

model_under = make_pipeline(
    SimpleImputer(strategy="median"), DecisionTreeClassifier(random_state=42)
)
model_under.fit(X_train_under, y_train_under) 

# Fit on `X_train_over`, `y_train_over`

model_over = make_pipeline(
    SimpleImputer(strategy="median"), DecisionTreeClassifier(random_state=42)
)
model_over.fit(X_train_over, y_train_over)  

## Evalaute

In [ ]:
## Calculate training and test accuracy for your three models.

for m in [model_reg, model_under, model_over]:
    acc_train = m.score(X_train, y_train)
    acc_test = m.score(X_test, y_test)

    print("Training Accuracy:", round(acc_train, 4))
    print("Test Accuracy:", round(acc_test, 4))

In [ ]:
##  Plot a confusion matrix that shows how your best model performs on your validation set.

ConfusionMatrixDisplay.from_estimator(model_reg, X_test, y_test);

In [ ]:
## Determine the depth of the decision tree in model_over.

depth = model_over.named_steps["decisiontreeclassifier"].get_depth()
print(depth)

In [ ]:
## Create a horizontal bar chart with the 15 most important features for model_over. Be sure to label your x-axis "Gini Importance".

# Get importances

importances = model_over.named_steps["decisiontreeclassifier"].feature_importances_

# Put importances into a Series
feat_imp = pd.Series(importances, index=X_train_over.columns).sort_values()
# Plot series
feat_imp.tail(15).plot(kind="barh")
plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.title("model_over Feature Importance");

In [ ]:
##  Using a context manager, save your best-performing model to a a file named "model-5-2.pkl".

# Save your model as `"model-5-2.pkl"`
with open("model-5-2.pkl", "wb") as f:
    pickle.dump(model_over, f)

In [ ]:
## Make sure you've saved your model correctly by loading "model-5-2.pkl" and assigning to the variable loaded_model.

# Load `"model-5-2.pkl"`
with open("model-5-2.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print(loaded_model)

In [ ]:
## Add Predictions to a DataFrame of new dataset
import pandas as pd

# 1. Load new dataset
new_data = pd.read_csv("new_dataset.csv")

# 2. Make sure features are in the correct order
new_data_features = new_data[loaded_model.feature_names_in_]

# 3. Generate predictions
predictions = loaded_model.predict(new_data_features)

# 4. Create a DataFrame with predictions
# Option 1: Add to original data
new_data_with_predictions = new_data.copy()
new_data_with_predictions["Predicted_Bankrupt"] = predictions

# Option 2: Just predictions (if you want it separate)
predictions_df = pd.DataFrame(predictions, columns=["Predicted_Bankrupt"])
